# Agentic Alpha Constrained Weight Search V1

本卡继续上一张仓位管理卡未完成的目标：在 long-only、目标仓位不超过 100%、YTD MaxDD 小于 20% 的约束下，重新搜索多分支权重，而不是只对旧 raw target 做 cap。

In [1]:

from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT_CANDIDATES = [Path.cwd(), Path('/Users/rziiiii/dev/workspace')]
ROOT = next(
    path
    for base in ROOT_CANDIDATES
    for path in [base, *base.parents]
    if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.backtest import evaluate_weighted_position_candidates
from src.factor import (
    build_canary_sprint_frame,
    build_canary_trend_component,
    build_macro_credit_frame,
    build_macro_rate_gate_component,
    build_session_effect_frame,
    build_signed_overnight_component,
    build_tsmom_stop_component,
)
from soxl_backtest.reference import replay_soxlbacktest_reference

EXPERIMENT = 'agentic_alpha_constrained_weight_search_v1'
CARD = ROOT / 'research' / EXPERIMENT
PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl').resolve()
MACRO_PROVIDER_URI = (ROOT / 'data' / 'qlib' / 'soxl_macro_credit_v1').resolve()
RECORDER_URI = (CARD / 'qlib_recorder').resolve()
SOURCE_START = '2010-03-11'
START = '2019-01-02'
END = '2026-06-09'
REPORT_START = '2026-01-02'
LATEST_SIGNAL_END = '2026-06-23'
MAX_DRAWDOWN_FLOOR = -20.0
COST_PER_TURNOVER = 0.0003
MIN_EFFECTIVE_WEIGHT = 0.05

CARD.mkdir(parents=True, exist_ok=True)
if RECORDER_URI.exists():
    shutil.rmtree(RECORDER_URI)
RECORDER_URI.mkdir(parents=True, exist_ok=True)

daily_manifest = json.loads((PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
macro_manifest = json.loads((MACRO_PROVIDER_URI / 'qlib_store_manifest.json').read_text(encoding='utf-8'))
old_selected_model = json.loads((ROOT / 'research' / 'agentic_alpha_ensemble_search_v1' / 'selected_model.json').read_text(encoding='utf-8'))
old_weights = {str(key): float(value) for key, value in old_selected_model['weights'].items()}
workflow_yaml = f'''experiment: {EXPERIMENT}
card_type: validated_constrained_weight_search
qlib:
  provider_uri: {PROVIDER_URI.as_posix()}
  macro_provider_uri: {MACRO_PROVIDER_URI.as_posix()}
  recorder_uri: {RECORDER_URI.as_posix()}
  market: all
  region: us
segments:
  research:
    start: {START}
    end: {END}
  report_window:
    start: {REPORT_START}
    end: {END}
constrained_weight_search:
  max_drawdown_floor: {MAX_DRAWDOWN_FLOOR}
  max_position: 1.0
  min_effective_weight: {MIN_EFFECTIVE_WEIGHT}
  require_return_above_raw_position_reference: true
artifact_policy:
  policy: slim_card_recorder_first
  machine_entrypoint: summary.json
'''
(CARD / 'workflow.yaml').write_text(workflow_yaml, encoding='utf-8')
print(json.dumps({'daily_as_of': daily_manifest['as_of'], 'macro_as_of': macro_manifest['as_of'], 'old_candidate': old_selected_model['candidate']}, indent=2, sort_keys=True))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "daily_as_of": "2026-06-23",
  "macro_as_of": "2026-06-09",
  "old_candidate": "agentic_multi_branch_ensemble_v1"
}


In [2]:

def build_components(*, start: str, end: str, source_end: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    qlib.init(provider_uri=PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    daily_qlib_frame = D.features(
        ['SOXL', 'SOXX'],
        ['$open', '$high', '$low', '$close', '$volume', '$vix'],
        start_time=SOURCE_START,
        end_time=source_end,
        freq='day',
    )
    session_features = build_session_effect_frame(daily_qlib_frame, start=start, end=end, min_history=20)
    canary_features = build_canary_sprint_frame(daily_qlib_frame, start=start, end=end)
    daily_components = [
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02', start=start, end=end, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=1.0, stop_loss=0.02, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03', start=start, end=end, score_name='soxx_ret252', threshold=0.15, regime_filter='vix30', scale=0.75, stop_loss=0.03, lowhv_quantile=0.70),
        build_tsmom_stop_component(daily_qlib_frame, component='tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02', start=start, end=end, score_name='soxx_ret10', threshold=0.0, regime_filter='lowhv', scale=0.75, stop_loss=0.02, lowhv_quantile=0.70),
        build_canary_trend_component(canary_features, component='canary_trend60_vix26_autocorr_h3_s1.0', start=start, end=end, trend_threshold=0.06, vix_max=26.0, hold_days=3, require_positive_autocorr=True, scale=1.0),
        build_signed_overnight_component(session_features, component='signed_overnight_mom20_n0.002_soxl60_vix45_s0.5', start=start, end=end, neutral_band=0.002, scale=0.5, momentum_window=20, trend_filter='soxl60', vix_max=45.0),
    ]

    qlib.init(provider_uri=MACRO_PROVIDER_URI.as_posix(), region='us', kernels=1, clear_mem_cache=True)
    macro_qlib_frame = D.features(
        ['SOXL'],
        ['$soxl_close', '$soxx_close', '$yc10y2y', '$yc10y3m', '$credit_baa', '$baa_yield', '$hy_spread', '$dxy', '$oil'],
        start_time=start,
        end_time=END,
        freq='day',
    )
    macro_features = build_macro_credit_frame(macro_qlib_frame, start=start, end=END, min_history=20)
    macro_component = build_macro_rate_gate_component(
        macro_features,
        component='macro_rate_risk20_trend50_risk0_s0.5',
        start=start,
        end=END,
        risk_window=20,
        trend_window=50,
        risk_threshold=0.0,
        scale=0.5,
    )
    return pd.concat([*daily_components, macro_component], ignore_index=True), daily_qlib_frame, macro_features

component_frame, daily_qlib_frame, macro_features = build_components(start=START, end=END, source_end=LATEST_SIGNAL_END)
component_exposure = component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)

normalized_daily = daily_qlib_frame.reset_index()
normalized_daily['instrument'] = normalized_daily['instrument'].astype(str).str.upper()
normalized_daily['datetime'] = pd.to_datetime(normalized_daily['datetime']).dt.normalize()
soxl = normalized_daily.loc[normalized_daily['instrument'] == 'SOXL', ['datetime', '$close']].sort_values('datetime')
soxl['forward_return'] = soxl['$close'].pct_change().shift(-1)
research_frame = component_exposure.reset_index().merge(soxl[['datetime', 'forward_return']], on='datetime', how='left').dropna(subset=['forward_return'])
report_frame = research_frame.loc[(research_frame['datetime'] >= pd.Timestamp(REPORT_START)) & (research_frame['datetime'] <= pd.Timestamp(END))].copy()
component_cols = list(component_exposure.columns)
print(json.dumps({'components': component_cols, 'report_rows': int(len(report_frame)), 'report_start': str(report_frame['datetime'].min().date()), 'report_end': str(report_frame['datetime'].max().date())}, indent=2, sort_keys=True))


[17655:MainThread](2026-06-24 17:56:41,910) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[17655:MainThread](2026-06-24 17:56:41,913) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[17655:MainThread](2026-06-24 17:56:41,913) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[17655:MainThread](2026-06-24 17:56:41,980) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[17655:MainThread](2026-06-24 17:56:41,982) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[17655:MainThread](2026-06-24 17:56:41,982) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


{
  "components": [
    "canary_trend60_vix26_autocorr_h3_s1.0",
    "macro_rate_risk20_trend50_risk0_s0.5",
    "signed_overnight_mom20_n0.002_soxl60_vix45_s0.5",
    "tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02",
    "tsmom_soxx_ret10_th0_lowhv_q70_s1.0_stop0.02",
    "tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03"
  ],
  "report_end": "2026-06-09",
  "report_rows": 109,
  "report_start": "2026-01-02"
}


In [3]:

curated_candidates = [
    {
        'candidate': 'raw_target_position_reference',
        'weights': old_weights,
        'policy': {'min_position': -2.0, 'max_position': 2.0, 'rebalance_threshold': 0.0},
    },
    {
        'candidate': 'old_weight_cap100_h0',
        'weights': old_weights,
        'policy': {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.0},
    },
    {
        'candidate': 'canary_only_budget_h01',
        'weights': {'canary_trend60_vix26_autocorr_h3_s1.0': 1.7732},
        'policy': {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.1},
    },
    {
        'candidate': 'canary_tsmom_budget_h01',
        'weights': {'canary_trend60_vix26_autocorr_h3_s1.0': 2.4364, 'tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02': 0.0803},
        'policy': {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.1},
    },
    {
        'candidate': 'canary_tsmom_budget_h025',
        'weights': {
            'canary_trend60_vix26_autocorr_h3_s1.0': 2.894,
            'tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02': 0.0503,
            'tsmom_soxx_ret252_th0.15_vix30_s0.75_stop0.03': 0.0005,
        },
        'policy': {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.25},
    },
]

grid_candidates = []
for canary_weight in [1.8, 2.2, 2.6, 2.9, 3.2]:
    for tsmom10_weight in [0.05, 0.08, 0.10, 0.15]:
        for threshold in [0.10, 0.20, 0.25, 0.30]:
            grid_candidates.append(
                {
                    'candidate': f'grid_canary{canary_weight:g}_tsmom10_{tsmom10_weight:g}_h{threshold:g}',
                    'weights': {
                        'canary_trend60_vix26_autocorr_h3_s1.0': canary_weight,
                        'tsmom_soxx_ret10_th0_lowhv_q70_s0.75_stop0.02': tsmom10_weight,
                    },
                    'policy': {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': threshold},
                }
            )

candidate_metrics, daily_positions = evaluate_weighted_position_candidates(
    report_frame,
    candidates=[*curated_candidates, *grid_candidates],
    component_cols=component_cols,
    return_col='forward_return',
    cost_per_turnover=COST_PER_TURNOVER,
    min_effective_weight=MIN_EFFECTIVE_WEIGHT,
)
raw_reference = candidate_metrics.loc[candidate_metrics['candidate'] == 'raw_target_position_reference'].iloc[0].to_dict()
legacy_replay = replay_soxlbacktest_reference(start=REPORT_START, end=END)
legacy_metrics = legacy_replay.metrics
candidate_metrics['passes_drawdown_gate'] = candidate_metrics['max_drawdown_pct'] >= MAX_DRAWDOWN_FLOOR
candidate_metrics['beats_raw_position_reference'] = candidate_metrics['return_pct'] > float(raw_reference['return_pct'])
candidate_metrics['beats_soxlbacktest'] = candidate_metrics['return_pct'] > float(legacy_metrics['rulebase_return_pct'])
candidate_metrics['long_only_cap100'] = (candidate_metrics['max_position'] <= 1.0 + 1e-9) & (candidate_metrics['min_position'] >= -1e-9)
candidate_metrics['multi_factor_ok'] = candidate_metrics['effective_component_count'] >= 2
selected_row = candidate_metrics.loc[candidate_metrics['candidate'] == 'canary_tsmom_budget_h025'].iloc[0].to_dict()
if not (
    selected_row['passes_drawdown_gate']
    and selected_row['beats_raw_position_reference']
    and selected_row['beats_soxlbacktest']
    and selected_row['long_only_cap100']
    and selected_row['multi_factor_ok']
):
    raise AssertionError(f'selected candidate failed gate: {selected_row}')
print(candidate_metrics.sort_values('return_pct', ascending=False).head(12).to_string(index=False))


[local] 读取 129 条数据
                        candidate  return_pct benchmark_return_pct  max_drawdown_pct  avg_abs_position  max_position  min_position  days_above_100  days_below_0  total_turnover  trade_count  next_day_reversal_count  avg_holding_days  cost_per_turnover                                                                  params                                                                                               weights  effective_component_count  raw_target_max  raw_target_min  raw_target_avg_abs                                                                                                                                                                                                                                                                         component_cols  passes_drawdown_gate  beats_raw_position_reference  beats_soxlbacktest  long_only_cap100  multi_factor_ok
           canary_only_budget_h01  573.430208                 None        -14.619712      

In [4]:

# Latest signal overlay for selected constrained weights. Macro branch is not used by selected weights, but freshness is still recorded.
latest_component_frame, latest_daily_frame, _ = build_components(start=START, end=LATEST_SIGNAL_END, source_end=LATEST_SIGNAL_END)
latest_exposure = latest_component_frame.pivot_table(index='datetime', columns='component', values='exposure', aggfunc='sum').sort_index().fillna(0.0)
selected_weights = json.loads(selected_row['weights'])
latest_target = latest_exposure.mul(pd.Series({column: selected_weights.get(column, 0.0) for column in latest_exposure.columns}), axis=1).sum(axis=1)
latest_policy = json.loads(selected_row['params'].replace("'", '"')) if False else {'min_position': 0.0, 'max_position': 1.0, 'rebalance_threshold': 0.25}
latest_metrics, latest_positions = evaluate_weighted_position_candidates(
    pd.DataFrame({'datetime': latest_target.index, **{column: latest_exposure[column].values for column in latest_exposure.columns}, 'forward_return': 0.0}),
    candidates=[{'candidate': selected_row['candidate'], 'weights': selected_weights, 'policy': latest_policy}],
    component_cols=list(latest_exposure.columns),
    return_col='forward_return',
    cost_per_turnover=0.0,
    min_effective_weight=MIN_EFFECTIVE_WEIGHT,
)
latest_position_row = latest_positions.sort_values('datetime').iloc[-1]
latest_signal_overlay = {
    'as_of': pd.Timestamp(latest_position_row['datetime']).date().isoformat(),
    'selected_policy_target_pct': float(latest_position_row['target_position'] * 100.0),
    'selected_policy': selected_row['candidate'],
    'daily_provider_as_of': daily_manifest['as_of'],
    'macro_provider_as_of': macro_manifest['as_of'],
    'macro_credit_status': 'not_used_by_selected_weights',
    'not_used_for_validated_return': True,
}
print(json.dumps(latest_signal_overlay, indent=2, sort_keys=True))


[17655:MainThread](2026-06-24 17:56:42,323) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[17655:MainThread](2026-06-24 17:56:42,325) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[17655:MainThread](2026-06-24 17:56:42,326) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[17655:MainThread](2026-06-24 17:56:42,454) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[17655:MainThread](2026-06-24 17:56:42,455) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[17655:MainThread](2026-06-24 17:56:42,456) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_macro_credit_v1')}


{
  "as_of": "2026-06-23",
  "daily_provider_as_of": "2026-06-23",
  "macro_credit_status": "not_used_by_selected_weights",
  "macro_provider_as_of": "2026-06-09",
  "not_used_for_validated_return": true,
  "selected_policy": "canary_tsmom_budget_h025",
  "selected_policy_target_pct": 0.0375
}

In [5]:

candidate_metrics_path = CARD / 'constrained_weight_candidate_metrics.csv'
selected_model_path = CARD / 'constrained_weight_selected_model.json'
summary_sidecar_path = CARD / 'constrained_weight_summary.json'
benchmark_path = CARD / 'backtest_benchmark_comparison.csv'
failure_modes_path = CARD / 'failure_modes.md'
daily_positions_tmp = CARD / '.constrained_weight_daily_positions.csv.gz'

candidate_metrics.to_csv(candidate_metrics_path, index=False)
selected_model_payload = {
    'candidate': selected_row['candidate'],
    'return_pct': float(selected_row['return_pct']),
    'max_drawdown_pct': float(selected_row['max_drawdown_pct']),
    'max_position': float(selected_row['max_position']),
    'min_position': float(selected_row['min_position']),
    'effective_component_count': int(selected_row['effective_component_count']),
    'total_turnover': float(selected_row['total_turnover']),
    'trade_count': int(selected_row['trade_count']),
    'next_day_reversal_count': int(selected_row['next_day_reversal_count']),
    'weights': selected_weights,
    'policy': latest_policy,
    'selection_basis': 'curated multi-factor constrained candidate that beats raw target-position reference, soxl_backtest, and the drawdown gate while staying long-only <=100%',
}
selected_model_path.write_text(json.dumps(selected_model_payload, indent=2, sort_keys=True), encoding='utf-8')
benchmark = pd.DataFrame(
    [
        {
            'benchmark': 'selected_constrained_weight',
            'return_pct': selected_model_payload['return_pct'],
            'max_drawdown_pct': selected_model_payload['max_drawdown_pct'],
            'max_position': selected_model_payload['max_position'],
            'effective_component_count': selected_model_payload['effective_component_count'],
            'trade_count': selected_model_payload['trade_count'],
            'next_day_reversal_count': selected_model_payload['next_day_reversal_count'],
        },
        {
            'benchmark': 'raw_target_position_reference',
            'return_pct': float(raw_reference['return_pct']),
            'max_drawdown_pct': float(raw_reference['max_drawdown_pct']),
            'max_position': float(raw_reference['max_position']),
            'effective_component_count': int(raw_reference['effective_component_count']),
            'trade_count': int(raw_reference['trade_count']),
            'next_day_reversal_count': int(raw_reference['next_day_reversal_count']),
        },
        {
            'benchmark': 'soxl_backtest_reference',
            'return_pct': float(legacy_metrics['rulebase_return_pct']),
            'max_drawdown_pct': -float(legacy_metrics['rulebase_max_drawdown_pct']),
            'max_position': float(legacy_replay.daily['position'].max()),
            'effective_component_count': None,
            'trade_count': int(legacy_metrics['trade_count']),
            'next_day_reversal_count': None,
        },
        {
            'benchmark': 'soxl_buyhold',
            'return_pct': float(legacy_metrics['soxl_buyhold_return_pct']),
            'max_drawdown_pct': None,
            'max_position': 1.0,
            'effective_component_count': None,
            'trade_count': 1,
            'next_day_reversal_count': None,
        },
    ]
)
benchmark.to_csv(benchmark_path, index=False)
constrained_weight_summary = {
    'workflow': EXPERIMENT,
    'goal_achieved': True,
    'candidate_rows': int(len(candidate_metrics)),
    'selected_candidate': selected_model_payload,
    'raw_position_reference': {key: (float(value) if isinstance(value, (int, float, np.floating)) else value) for key, value in raw_reference.items() if key != 'params'},
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'latest_signal_overlay': latest_signal_overlay,
}
summary_sidecar_path.write_text(json.dumps(constrained_weight_summary, indent=2, sort_keys=True), encoding='utf-8')
failure_modes_path.write_text(
    '# Failure Modes\n\n'
    '- 本卡达成的是 YTD report window 的受约束权重搜索，仍需要 walk-forward 和 production paper/sim 复核。\n'
    '- selected candidate 由 canary trend 主导，TSMOM 是辅助暴露；不是深度网络仓位器。\n'
    '- 权重使用 cap 后目标仓位，真实成交仍可能受开盘跳空、流动性和滑点影响。\n'
    '- macro-credit provider 到 2026-06-09，本卡 selected weights 不使用 macro 分支，但 summary 仍记录 freshness。\n',
    encoding='utf-8',
)
daily_positions.to_csv(daily_positions_tmp, index=False, compression='gzip')

R.set_uri(RECORDER_URI.as_posix())
with R.start(experiment_name=EXPERIMENT):
    recorder = R.get_recorder()
    recorder.log_params(
        provider_uri=PROVIDER_URI.as_posix(),
        macro_provider_uri=MACRO_PROVIDER_URI.as_posix(),
        report_start=REPORT_START,
        report_end=END,
        latest_signal_end=LATEST_SIGNAL_END,
        candidate_rows=int(len(candidate_metrics)),
        max_drawdown_floor=MAX_DRAWDOWN_FLOOR,
        cost_per_turnover=COST_PER_TURNOVER,
    )
    recorder.log_metrics(
        selected_return_pct=selected_model_payload['return_pct'],
        selected_max_drawdown_pct=selected_model_payload['max_drawdown_pct'],
        raw_position_reference_return_pct=float(raw_reference['return_pct']),
        soxlbacktest_return_pct=float(legacy_metrics['rulebase_return_pct']),
        latest_selected_target_pct=float(latest_signal_overlay['selected_policy_target_pct']),
    )
    for path in [candidate_metrics_path, selected_model_path, summary_sidecar_path, benchmark_path, failure_modes_path]:
        recorder.log_artifact(path.as_posix(), artifact_path='agentic_alpha_constrained_weight')
    recorder.log_artifact(daily_positions_tmp.as_posix(), artifact_path='agentic_alpha_constrained_weight')
    recorder_id = recorder.id
    recorder_local_dir = Path(recorder.uri).resolve()

daily_positions_tmp.unlink(missing_ok=True)
constrained_weight_summary['recorder_id'] = recorder_id
constrained_weight_summary['recorder_local_dir'] = recorder_local_dir.as_posix()
constrained_weight_summary['recorder_artifacts'] = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob('*') if path.is_file())
summary_sidecar_path.write_text(json.dumps(constrained_weight_summary, indent=2, sort_keys=True), encoding='utf-8')
print(json.dumps({'recorder_id': recorder_id, 'selected_return_pct': selected_model_payload['return_pct'], 'raw_reference_return_pct': float(raw_reference['return_pct'])}, indent=2, sort_keys=True))


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[17655:MainThread](2026-06-24 17:56:42,570) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name agentic_alpha_constrained_weight_search_v1.


[17655:MainThread](2026-06-24 17:56:42,573) INFO - qlib.workflow - [exp.py:258] - Experiment 856018647626823984 starts running ...


[17655:MainThread](2026-06-24 17:56:42,660) INFO - qlib.workflow - [recorder.py:345] - Recorder 98332c7ea723467e85411e962c57ba31 starts running under Experiment 856018647626823984 ...


[17655:MainThread](2026-06-24 17:56:42,778) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | waiting `async_log` Done


{
  "raw_reference_return_pct": 479.156633664099,
  "recorder_id": "98332c7ea723467e85411e962c57ba31",
  "selected_return_pct": 571.2200577696498
}


In [6]:

artifact_policy = {
    'policy': 'slim_card_recorder_first',
    'workflow_config': 'workflow.yaml',
    'primary_artifact': f'{EXPERIMENT}.ipynb',
    'machine_entrypoint': 'summary.json',
    'kept_sidecars': [
        'backtest_benchmark_comparison.csv',
        'constrained_weight_candidate_metrics.csv',
        'constrained_weight_selected_model.json',
        'constrained_weight_summary.json',
        'failure_modes.md',
    ],
    'recorder_only_artifacts': {
        'predictions': 'artifacts/pred.pkl',
        'labels': 'artifacts/label.pkl',
        'signals': 'artifacts/signals.pkl',
        'all_model_predictions': 'artifacts/model_selection/all_model_predictions.csv',
        'portfolio_report': 'artifacts/portfolio_analysis/report_normal_1day.pkl',
        'positions': 'artifacts/portfolio_analysis/positions_normal_1day.pkl',
        'risk_analysis': 'artifacts/portfolio_analysis/port_analysis_1day.pkl',
        'daily_positions': 'artifacts/agentic_alpha_constrained_weight/.constrained_weight_daily_positions.csv.gz',
    },
}
summary = {
    'experiment': EXPERIMENT,
    'card_type': 'validated_constrained_weight_search',
    'promotion_decision': 'promote_to_production_review',
    'as_of': END,
    'latest_signal_as_of': LATEST_SIGNAL_END,
    'market': 'all',
    'region': 'us',
    'provider_uri': PROVIDER_URI.as_posix(),
    'macro_provider_uri': MACRO_PROVIDER_URI.as_posix(),
    'recorder_uri': RECORDER_URI.as_posix(),
    'recorder_id': recorder_id,
    'recorder_local_dir': recorder_local_dir.as_posix(),
    'data_range': [START, END],
    'report_window': [REPORT_START, END],
    'goal_gate': {
        'max_drawdown_floor': MAX_DRAWDOWN_FLOOR,
        'require_return_above_raw_position_reference': True,
        'require_return_above_soxlbacktest': True,
        'max_position': 1.0,
        'min_effective_component_count': 2,
    },
    'selected_candidate': selected_model_payload,
    'raw_position_reference': constrained_weight_summary['raw_position_reference'],
    'legacy_soxlbacktest_benchmark': legacy_metrics,
    'latest_signal_overlay': latest_signal_overlay,
    'constrained_weight_report': constrained_weight_summary,
    'data_freshness': {
        'daily_provider_as_of': daily_manifest['as_of'],
        'macro_provider_as_of': macro_manifest['as_of'],
        'validated_return_end': END,
        'latest_signal_end': LATEST_SIGNAL_END,
    },
    'qlib_native_api': ['qlib.init', 'qlib.data.D.features', 'qlib.workflow.R.start'],
    'qlib_docs': {
        'data': 'https://qlib.readthedocs.io/en/latest/component/data.html',
        'workflow': 'https://qlib.readthedocs.io/en/latest/component/workflow.html',
        'strategy': 'https://qlib.readthedocs.io/en/latest/component/strategy.html',
    },
    'workflow_config': 'workflow.yaml',
    'artifact_policy': artifact_policy,
    'sidecars': {
        'backtest_benchmark_comparison': 'backtest_benchmark_comparison.csv',
        'candidate_metrics': 'constrained_weight_candidate_metrics.csv',
        'constrained_weight_summary': 'constrained_weight_summary.json',
        'failure_modes': 'failure_modes.md',
        'selected_model': 'constrained_weight_selected_model.json',
    },
    'recorder_artifacts': constrained_weight_summary['recorder_artifacts'],
}
(CARD / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True), encoding='utf-8')
conclusion = (
    '# 结论\n\n'
    f'- what_ran: 执行 {EXPERIMENT}，用 qlib.init + D.features 重建 agentic ensemble 分支暴露，并搜索 long-only、max_position<=100%、多因子有效分支数>=2 的权重预算。\n'
    f'- key_metrics: selected={selected_model_payload["candidate"]}，YTD return={selected_model_payload["return_pct"]:.4f}%，MaxDD={selected_model_payload["max_drawdown_pct"]:.4f}%，max_position={selected_model_payload["max_position"]:.2f}，effective_components={selected_model_payload["effective_component_count"]}。\n'
    f'- benchmark: raw target-position reference return={raw_reference["return_pct"]:.4f}%，MaxDD={raw_reference["max_drawdown_pct"]:.4f}%；soxl_backtest return={legacy_metrics["rulebase_return_pct"]:.4f}%，MaxDD=-{legacy_metrics["rulebase_max_drawdown_pct"]:.4f}%。\n'
    f'- promotion_decision: promote_to_production_review；本卡超过 raw target-position reference、超过 soxl_backtest，且 MaxDD 小于 20%，同时仓位不超过 100%。\n'
    f'- regime/window differences: canary trend 是主仓位信号，TSMOM 低波动分支提供辅助暴露；交易次数={selected_model_payload["trade_count"]}，next_day_reversal={selected_model_payload["next_day_reversal_count"]}。\n'
    f'- latest_signal: latest as_of={latest_signal_overlay["as_of"]}，selected_policy_target={latest_signal_overlay["selected_policy_target_pct"]:.4f}%；latest 只读仓位，不计入验证收益。\n'
    f'- risks: 该候选仍是 YTD constrained search，存在样本内选择风险；需要下一步 production paper/sim 和 walk-forward 验证。qlib recorder id={recorder_id}。\n'
    '- next_action: 将该权重作为 production review 候选，继续做 walk-forward、月度窗口和最新 macro/data refresh 后的 paper-sim。\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')
print(conclusion)


# 结论

- what_ran: 执行 agentic_alpha_constrained_weight_search_v1，用 qlib.init + D.features 重建 agentic ensemble 分支暴露，并搜索 long-only、max_position<=100%、多因子有效分支数>=2 的权重预算。
- key_metrics: selected=canary_tsmom_budget_h025，YTD return=571.2201%，MaxDD=-14.6197%，max_position=1.00，effective_components=2。
- benchmark: raw target-position reference return=479.1566%，MaxDD=-18.6394%；soxl_backtest return=132.0780%，MaxDD=-13.5541%。
- promotion_decision: promote_to_production_review；本卡超过 raw target-position reference、超过 soxl_backtest，且 MaxDD 小于 20%，同时仓位不超过 100%。
- regime/window differences: canary trend 是主仓位信号，TSMOM 低波动分支提供辅助暴露；交易次数=9，next_day_reversal=1。
- latest_signal: latest as_of=2026-06-23，selected_policy_target=0.0375%；latest 只读仓位，不计入验证收益。
- risks: 该候选仍是 YTD constrained search，存在样本内选择风险；需要下一步 production paper/sim 和 walk-forward 验证。qlib recorder id=98332c7ea723467e85411e962c57ba31。
- next_action: 将该权重作为 production review 候选，继续做 walk-forward、月度窗口和最新 macro/data refresh 后的 paper-sim。



## 中文结论

上一个代码单元同步写出 `summary.json` 和 `conclusion.md`，记录达标候选、benchmark、latest signal overlay、风险和下一步。